In [ ]:
from infostop import Infostop
import numpy as np
import polars as pl
from pyproj import Transformer

import yaml

In [46]:
with open("../config.yaml") as fp:
    config = yaml.safe_load(fp)


In [ ]:
# stop detection process
with open("05_detect_stops_config.yaml") as fp:
    params = yaml.safe_load(fp)["infostop"]

model = Infostop(
    r1=params["r1"],
    r2=params["r2"],
    min_staying_time=params["min_staying_time"],
    max_time_between=params["max_time_between"],
    min_size=params["min_size"],
    distance_metric=params["distance_metric"],
)

In [ ]:
transformer = Transformer.from_crs(4326, config["crs"])

In [3]:
def initial_filter(df: pl.DataFrame, min_pings: int) -> pl.DataFrame:
    """Keep only devices with at least min_pings pings."""
    valid_devices = (
        df.group_by("device_id")
        .len()
        .filter(pl.col("len") >= min_pings)["device_id"]
        .to_list()
    )
    return df.filter(pl.col("device_id").is_in(valid_devices))


In [5]:
data = pl.read_parquet(
    "/mnt/common-hdd/group-dirs/tkom-2026-project/partitioned/00/pings.parquet"
)


In [ ]:
filtered = initial_filter(data, min_pings=config["filtering"]["min_pings_per_day"])
filtered = filtered.with_columns(
    pl.col("timestamp").dt.epoch(time_unit="s").alias("epoch")
)

In [17]:
# run infostop per device — data already sorted, no re-sort needed in loop
# catch devices with no stop events (all pings labelled -1)
results = []
for device_id, group in filtered.group_by("device_id"):
    coords = group[["latitude", "longitude", "epoch"]].to_numpy()
    try:
        labels = model.fit_predict(coords)
    except Exception:
        labels = np.full(len(coords), -1)
    group_ = group.with_columns(place_label=labels)

    results.append(group_)

clustered = pl.concat(results)
clustered = clustered.sort(["device_id", "timestamp"])

n_stopped = (clustered["place_label"] >= 0).sum()
print(
    f"Stop pings:    {n_stopped:,}  ({n_stopped / len(clustered):.1%} of filtered pings)"
)
print(f"Unique stops:  {clustered['place_label'].clip(lower_bound=0).n_unique():,}")
clustered.head(3)


Stop pings:    9,474,653  (61.3% of filtered pings)
Unique stops:  393


timestamp,device_id,device_aid_type,latitude,longitude,epoch,place_label
"datetime[μs, Europe/Budapest]",i64,str,f64,f64,i64,i64
2024-10-02 02:37:09 CEST,1,"""AAID""",47.47992,19.074155,1727829429,2
2024-10-02 03:17:49 CEST,1,"""AAID""",47.479995,19.074221,1727831869,2
2024-10-02 03:26:50 CEST,1,"""AAID""",47.476752,19.075154,1727832410,-1


In [ ]:
clustered.drop("epoch").write_parquet(
    "/mnt/common-hdd/group-dirs/tkom-2026-project/clustered_pings/00.parquet"
)

In [7]:
clustered = pl.read_parquet(
    "/mnt/common-hdd/group-dirs/tkom-2026-project/clustered_pings/00.parquet"
)

In [ ]:
clustered = clustered.with_columns(
    pl.when(pl.col("place_label") >= 0)
    .then(
        (
            (pl.col("place_label") >= 0)
            & (pl.col("place_label").shift(1).over("device_id").fill_null(-1) < 0)
        )
        .cast(pl.Int32)
        .cum_sum()
        .over("device_id")
    )
    .otherwise(-1)
    .alias("stop_id")
)

In [29]:
place_lable_jt = (
    clustered.filter(pl.col.stop_id >= 0)[["device_id", "stop_id", "place_label"]]
    .unique()
    .sort(["device_id", "stop_id"])
)

In [ ]:
clustered.filter((pl.col.stop_id == 4) & (pl.col.device_id == 1))

timestamp,device_id,device_aid_type,latitude,longitude,place_label,stop_id,latitude_proj,longitude_proj
"datetime[μs, Europe/Budapest]",i64,str,f64,f64,i64,i32,f64,f64
2024-10-02 08:57:56 CEST,1,"""AAID""",47.509079,19.151371,20,4,657828.482382,240577.300753
2024-10-02 09:17:56 CEST,1,"""AAID""",47.509079,19.151371,20,4,657828.482382,240577.300753
2024-10-02 09:52:28 CEST,1,"""AAID""",47.509079,19.151371,20,4,657828.482382,240577.300753
2024-10-02 09:52:53 CEST,1,"""AAID""",47.661187,19.076368,3,4,652172.708182,257483.68671
2024-10-02 10:23:47 CEST,1,"""AAID""",47.661213,19.076304,3,4,652167.900066,257486.575707
2024-10-02 10:33:00 CEST,1,"""AAID""",47.66122,19.07622,3,4,652161.590488,257487.35174


In [ ]:
projected = transformer.transform(clustered["latitude"], clustered["longitude"])
clustered = clustered.with_columns(
    latitude_proj=projected[0], longitude_proj=projected[1]
)

In [ ]:
stops = (
    clustered.filter(pl.col.stop_id >= 0)
    .group_by(["device_id", "stop_id"])
    .agg(
        interval_start=pl.col.timestamp.min(),
        interval_stop=pl.col.timestamp.max(),
        interval_lenght=pl.col.timestamp.max() - pl.col.timestamp.min(),
        latitude_mean=pl.col.latitude.mean(),
        longitude_mean=pl.col.longitude.mean(),
        latitude_std=pl.col.latitude_proj.std().round(),
        longitude_std=pl.col.longitude_proj.std().round(),
        number_of_pings=pl.col.longitude.count(),
    )
    .join(place_lable_jt, on=["device_id", "stop_id"])
    .sort(["device_id", "stop_id"])
)
stops.write_parquet("/mnt/common-hdd/group-dirs/tkom-2026-project/stops/00.parquet")
stops

device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label
i64,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64
1,1,2024-10-02 02:37:09 CEST,2024-10-02 03:17:49 CEST,40m 40s,47.479957,19.074188,4.0,6.0,2,2
1,2,2024-10-02 05:30:54 CEST,2024-10-02 06:30:49 CEST,59m 55s,47.480143,19.073989,10.0,65.0,3,2
1,3,2024-10-02 07:03:24 CEST,2024-10-02 07:35:29 CEST,32m 5s,47.661076,19.076359,2.0,14.0,5,3
1,4,2024-10-02 08:57:56 CEST,2024-10-02 10:33:00 CEST,1h 35m 4s,47.585143,19.113834,3101.0,9261.0,6,20
1,4,2024-10-02 08:57:56 CEST,2024-10-02 10:33:00 CEST,1h 35m 4s,47.585143,19.113834,3101.0,9261.0,6,3
…,…,…,…,…,…,…,…,…,…,…
6584,144,2024-11-17 08:23:14 CET,2024-11-17 09:17:42 CET,54m 28s,47.606786,19.271004,13.0,22.0,9,3
6584,145,2024-11-17 09:45:35 CET,2024-11-17 15:51:24 CET,6h 5m 49s,47.606837,19.2709,5.0,13.0,15,3
6584,146,2024-11-17 17:57:33 CET,2024-11-17 21:46:42 CET,3h 49m 9s,47.606707,19.270745,32.0,21.0,13,3


In [ ]:
stops.filter(pl.col.device_id == 1)

device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label
i64,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64
1,1,2024-10-02 02:37:09 CEST,2024-10-02 03:17:49 CEST,40m 40s,47.479957,19.074188,4.0,6.0,2,2
1,2,2024-10-02 05:30:54 CEST,2024-10-02 06:30:49 CEST,59m 55s,47.480143,19.073989,10.0,65.0,3,2
1,3,2024-10-02 07:03:24 CEST,2024-10-02 07:35:29 CEST,32m 5s,47.661076,19.076359,2.0,14.0,5,3
1,4,2024-10-02 08:57:56 CEST,2024-10-02 10:33:00 CEST,1h 35m 4s,47.585143,19.113834,3101.0,9261.0,6,20
1,4,2024-10-02 08:57:56 CEST,2024-10-02 10:33:00 CEST,1h 35m 4s,47.585143,19.113834,3101.0,9261.0,6,3
…,…,…,…,…,…,…,…,…,…,…
1,70,2025-12-03 03:26:48 CET,2025-12-03 03:46:49 CET,20m 1s,47.476992,19.075928,2.0,0.0,2,5
1,71,2025-12-03 09:50:52 CET,2025-12-03 10:05:32 CET,14m 40s,47.498492,19.040361,5.0,16.0,2,9
1,72,2025-12-03 15:28:13 CET,2025-12-03 16:06:38 CET,38m 25s,47.479361,19.074001,17.0,1.0,3,2
